# Solvers

> Sampling-based planners over discrete multi-agent actions, joint or per agent.

In [ ]:
#| default_exp planning.solver

In [ ]:
#| hide
from nbdev.showdoc import *

The solvers keep one **categorical distribution per (time step, agent)** over the `K` discrete
actions, sample candidate joint plans from them (Gumbel-max), score the candidates with a
`Costable`, and refit the distributions. They pass candidates to the cost as one-hot vectors
``(B, S, horizon, num_agents, K)`` and return the planned actions ``(n_envs, horizon, num_agents)``.

| `mode` | cost | refit |
|---|---|---|
| `'joint'` (centralised) | `(B, S)` (a per-agent cost is summed) | from the best **joint** plans |
| `'per_agent'` (decentralised) | `(B, S, num_agents)` | each agent from the plans best for **its own** cost |

In both modes the agents' actions are rolled out together, so each agent's cost accounts for
what the others do. `CategoricalCEMSolver` refits from the `topk` best candidates;
`CategoricalMPPISolver` from all candidates, weighted by ``softmax(-cost / temperature)``.

In [ ]:
#| export
# Adapted from stable-worldmodel's planning package (https://github.com/galilai-group/stable-worldmodel):
#
# MIT License
#
# Copyright (c) 2026 GalilAI-group
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

from __future__ import annotations

import time
from typing import Any, Protocol, runtime_checkable

import gymnasium as gym
import numpy as np
import torch

from stable_marl.protocols import Constrainable, Costable

In [ ]:
#| export
@runtime_checkable
class Solver(Protocol):
    "Protocol of model-based planning solvers (as in stable-worldmodel)."

    def configure(self, *, action_space: gym.Space, n_envs: int, config: Any) -> None:
        "Configure the solver for the joint action space of one env, `n_envs` envs and a :class:`PlanConfig`."
        ...

    @property
    def action_dim(self) -> int: ...

    @property
    def n_envs(self) -> int: ...

    @property
    def horizon(self) -> int: ...

    def solve(self, info_dict: dict, init_action: torch.Tensor | None = None) -> dict:
        "Plan; returns at least ``{'actions': (n_envs, horizon, num_agents), 'costs': [...]}``."
        ...

In [ ]:
#| export
PLANNING_MODES = ('joint', 'per_agent')


class CategoricalCEMSolver:
    """
    Cross-entropy method over discrete multi-agent actions.

    Parameters
    ----------
    cost : Costable
        What to minimise (e.g. a :class:`ShootingCostEvaluator`)
    batch_size : int
        Envs planned for at once
    num_samples : int
        Candidate plans per env and iteration
    n_steps : int
        Iterations
    topk : int
        Elite candidates the distributions are refit from
    smoothing : float
        Added to the refit probabilities, so actions never become impossible
    alpha : float
        Momentum of the probabilities (0: replaced at each iteration)
    mode : 'joint' or 'per_agent'
        Centralised planning of the joint plan, or each agent for its own cost (see above)
    """
    def __init__(
        self,
        cost: Costable,
        batch_size: int = 1,
        num_samples: int = 300,
        n_steps: int = 30,
        topk: int = 30,
        smoothing: float = 0.0,
        alpha: float = 0.0,
        mode: str = 'joint',
        device: str | torch.device = 'cpu',
        seed: int = 1234,
        verbose: bool = False):
        if mode not in PLANNING_MODES:
            raise ValueError(f"mode must be one of {PLANNING_MODES}, got {mode!r}")
        self.cost, self.batch_size, self.num_samples, self.n_steps = cost, batch_size, num_samples, n_steps
        self.topk, self.smoothing, self.alpha, self.mode = topk, smoothing, alpha, mode
        self.device, self.verbose = device, verbose
        self.torch_gen = torch.Generator(device=device).manual_seed(seed)
        try:
            self._dtype = next(cost.parameters()).dtype
        except (AttributeError, StopIteration):
            self._dtype = torch.float32

    def configure(self, *, action_space: gym.Space, n_envs: int, config: Any):
        "`action_space`: the joint action space of one env, MultiDiscrete with one entry per agent."
        if isinstance(action_space, gym.spaces.Discrete):
            nvec = np.array([action_space.n])
        elif isinstance(action_space, gym.spaces.MultiDiscrete):
            nvec = np.asarray(action_space.nvec).reshape(-1)
        else:
            raise TypeError(f"Action space must be MultiDiscrete or Discrete, got {type(action_space)}")
        if len(set(nvec.tolist())) != 1:
            raise ValueError(f"All agents must have the same number of actions, got {nvec}")
        if getattr(config, 'action_block', 1) != 1:
            raise NotImplementedError("action_block > 1 is not supported yet")
        self._action_space, self._n_envs, self._config = action_space, n_envs, config
        self._num_agents, self._num_actions = len(nvec), int(nvec[0])

    @property
    def n_envs(self) -> int:
        return self._n_envs

    @property
    def num_agents(self) -> int:
        return self._num_agents

    @property
    def action_dim(self) -> int:
        "Number of discrete actions of each agent (size of the one-hot candidates)."
        return self._num_actions

    @property
    def horizon(self) -> int:
        return self._config.horizon

    @property
    def dtype(self) -> torch.dtype:
        return self._dtype

    def __call__(self, *args, **kwargs) -> dict:
        return self.solve(*args, **kwargs)

    def init_probs(self, n_envs: int) -> torch.Tensor:
        "Uniform probabilities, (n_envs, horizon, num_agents, K)."
        K = self._num_actions
        return torch.full((n_envs, self.horizon, self._num_agents, K), 1.0 / K, dtype=self.dtype, device=self.device)

    def _sample_indices(self, probs: torch.Tensor) -> torch.Tensor:
        "Gumbel-max samples of (B, H, A, K) probabilities: (B, num_samples, H, A)."
        B, H, A, K = probs.shape
        log_probs = probs.clamp_min(1e-10).log().unsqueeze(1).expand(B, self.num_samples, H, A, K)
        u = torch.rand(log_probs.shape, generator=self.torch_gen, device=self.device, dtype=self.dtype).clamp_min(1e-10)
        return (log_probs - (-u.log()).log()).argmax(dim=-1)

    def _costs(self, costs: torch.Tensor, B: int) -> torch.Tensor:
        "Validate the cost shape and bring it to (B, S, A) (a joint cost is broadcast over agents)."
        S, A = self.num_samples, self._num_agents
        if not isinstance(costs, torch.Tensor):
            raise TypeError(f"Expected the cost to be a torch.Tensor, got {type(costs)}")
        if self.mode == 'per_agent':
            if costs.shape != (B, S, A):
                raise ValueError(f"mode='per_agent' needs a per-agent cost of shape {(B, S, A)}, got {tuple(costs.shape)}")
            return costs
        if costs.shape == (B, S, A):
            costs = costs.sum(dim=-1)
        if costs.shape != (B, S):
            raise ValueError(f"mode='joint' needs a cost of shape {(B, S)} (or {(B, S, A)}), got {tuple(costs.shape)}")
        return costs.unsqueeze(-1).expand(B, S, A)

    def _weights(self, costs: torch.Tensor) -> torch.Tensor:
        "Refit weights of the candidates, (B, S, A) summing to 1 over S: uniform over the top-k."
        _, top = torch.topk(costs, k=self.topk, dim=1, largest=False)
        weights = torch.zeros_like(costs)
        weights.scatter_(1, top, 1.0 / self.topk)
        return weights

    def _expand(self, info_dict: dict, start: int, end: int) -> dict:
        "The infos of envs [start, end), with a candidate axis of size num_samples after the env axis."
        B, out = end - start, {}
        for k, v in info_dict.items():
            v = v[start:end] if isinstance(v, (torch.Tensor, np.ndarray, list)) else v
            if torch.is_tensor(v):
                v = v.to(device=self.device, dtype=self.dtype if v.is_floating_point() else None)
                v = v.unsqueeze(1).expand(B, self.num_samples, *v.shape[1:])
            elif isinstance(v, np.ndarray):
                v = np.repeat(v[:, None], self.num_samples, axis=1)
            out[k] = v
        return out

    @torch.inference_mode()
    def solve(self, info_dict: dict, init_action: Any = None) -> dict:
        """
        Plan for every env of `info_dict`. Returns ``actions`` (n_envs, horizon, num_agents),
        ``costs`` (the final mean elite cost of each env: a float, or one per agent in
        ``'per_agent'`` mode) and ``probs``. `init_action` is accepted for API parity and ignored.
        """
        start_time = time.time()
        n_envs = len(next(v for v in info_dict.values() if isinstance(v, (torch.Tensor, np.ndarray, list))))
        probs = self.init_probs(n_envs)
        final_costs = []
        for start in range(0, n_envs, self.batch_size):
            end = min(start + self.batch_size, n_envs)
            B, batch_probs = end - start, probs[start:end]
            infos = self._expand(info_dict, start, end)
            for step in range(self.n_steps):
                indices = self._sample_indices(batch_probs)
                indices[:, 0] = batch_probs.argmax(dim=-1)            # the current mode is always a candidate
                one_hot = torch.nn.functional.one_hot(indices, num_classes=self._num_actions).to(self.dtype)
                costs = self._costs(self.cost.get_cost(dict(infos), one_hot), B)
                weights = self._weights(costs)
                new_probs = torch.einsum('bsa,bshak->bhak', weights, one_hot)
                if self.smoothing > 0:
                    new_probs = (new_probs + self.smoothing) / (new_probs + self.smoothing).sum(dim=-1, keepdim=True)
                batch_probs = self.alpha * batch_probs + (1 - self.alpha) * new_probs if self.alpha > 0 else new_probs
            elite_costs = (weights * costs).sum(dim=1)                # (B, A)
            final_costs += elite_costs.tolist() if self.mode == 'per_agent' else elite_costs[:, 0].tolist()
            probs[start:end] = batch_probs
        if self.verbose:
            print(f"{type(self).__name__} solve time: {time.time() - start_time:.4f} seconds")
        return {'actions': probs.argmax(dim=-1).cpu(), 'costs': final_costs, 'probs': [probs.cpu()]}

In [ ]:
#| export
class CategoricalMPPISolver(CategoricalCEMSolver):
    """
    MPPI-style refit: every candidate weighted by ``softmax(-cost / temperature)`` over the
    samples (per agent in ``'per_agent'`` mode), instead of the top-k only.
    """
    def __init__(self, cost: Costable, temperature: float = 1.0, **kwargs):
        super().__init__(cost, **kwargs)
        self.temperature = temperature

    def _weights(self, costs: torch.Tensor) -> torch.Tensor:
        return torch.softmax(-(costs - costs.min(dim=1, keepdim=True).values) / self.temperature, dim=1)

### Tests

In [ ]:
from types import SimpleNamespace

class TargetCost:
    "Cost of a candidate: number of (step, agent) actions differing from a target plan."
    def __init__(self, target, per_agent=False):
        self.target, self.per_agent = target, per_agent        # target: (H, A) action indices
    def get_cost(self, info_dict, candidates):                 # candidates: (B, S, H, A, K)
        wrong = (candidates.argmax(-1) != self.target).float().sum(dim=2)   # (B, S, A)
        return wrong if self.per_agent else wrong.sum(-1)
    def criterion(self, info_dict, candidates):
        return self.get_cost(info_dict, candidates)

target = torch.tensor([[0, 3], [2, 2], [1, 0], [3, 1]])      # horizon 4, 2 agents, 4 actions
space = gym.spaces.MultiDiscrete([4, 4])
for solver_cls in (CategoricalCEMSolver, CategoricalMPPISolver):
    for mode in PLANNING_MODES:
        kwargs = dict(temperature=0.5) if solver_cls is CategoricalMPPISolver else dict(topk=20)
        solver = solver_cls(TargetCost(target, per_agent=mode == 'per_agent'), batch_size=3,
                            num_samples=200, n_steps=10, mode=mode, **kwargs)
        solver.configure(action_space=space, n_envs=3, config=SimpleNamespace(horizon=4, action_block=1))
        assert isinstance(solver, Solver)
        out = solver.solve({'x': torch.zeros(3, 1)})
        assert out['actions'].shape == (3, 4, 2) and (out['actions'] == target).all(), (solver_cls, mode)
        assert len(out['costs']) == 3

joint_only = CategoricalCEMSolver(TargetCost(target), num_samples=10, n_steps=1, topk=2, mode='per_agent')
joint_only.configure(action_space=space, n_envs=1, config=SimpleNamespace(horizon=4, action_block=1))
try:
    joint_only.solve({'x': torch.zeros(1, 1)}); raise AssertionError
except ValueError as e:
    assert 'per-agent cost' in str(e)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()